In [16]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))
load_dotenv()

True

In [17]:
from src.profiling import generate_queries, profile_to_context
from src.retrieval import load_retrieval_system, retrieve
from src.generation import generate_playlist
from src.evaluation import (
    precision_at_k, recall, mrr,
    historical_plausibility, llm_judge,
    create_human_eval_sheet, compute_inter_rater
)
from configs.prompts import GENERATION_PROMPT, BASELINE_PROMPT

import json
import time
import pandas as pd
from openai import OpenAI

client = OpenAI()

In [21]:
import os

print(os.listdir("../data/index"))
print(os.listdir("../data/processed"))

['songs.index', 'knowledge_base.csv', 'bm25.pkl']
['val_profiles.json', 'test_profiles.json']


In [22]:
faiss_index, df, bm25 = load_retrieval_system(
    "../data/index/songs.index",
    "../data/index/knowledge_base.csv",
    "../data/index/bm25.pkl"
)

print(f"Knowledge base loaded: {len(df)} songs")

patient = {
    "name": "James Wilson",
    "gender": "Male",
    "birth_year": 1948,
    "hometown": "Detroit, Michigan",
    "cultural_background": "African American",
    "life_events": [
        {"year": 1966, "event": "Graduated from Cass Tech High School"},
        {"year": 1968, "event": "Drafted into the Vietnam War"},
        {"year": 1971, "event": "Married Dorothy in Detroit"},
        {"year": 1975, "event": "First child born"},
    ]
}

retrieved, queries = profile_to_context(patient, faiss_index, bm25, df)
print(f"Generated {len(queries)} queries")
print(f"Retrieved {len(retrieved)} unique songs")

result = generate_playlist(patient, retrieved, GENERATION_PROMPT)

print("\n=== PLAYLIST ===")
for song in result["playlist"]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")
    print(f"     Reason: {song['relevance']}\n")

print("\n=== CAREGIVER CARDS ===")
for card in result["caregiver_cards"]:
    print(f"  Song: {card['song']}")
    print(f"  Prompt: {card['prompt']}\n")
    

Knowledge base loaded: 2381 songs
Generated 8 queries:
  - popular Motown hits in Detroit 1963-1973
  - top R&B songs 1966 Detroit
  - Vietnam War era songs popular among African American soldiers 1968
  - wedding songs popular in Detroit 1971
  - soul and funk hits 1975 Detroit
  - African American male favorite artists 1963-1973
  - graduation songs 1966 Detroit
  - Detroit radio hits 1968
Generated 8 queries
Retrieved 20 unique songs

=== PLAYLIST ===
  1. Respect — Aretha Franklin (1967)
     Reason: This song is by an iconic African American artist from Detroit, which is James's hometown. It was released during his formative years and represents cultural pride and empowerment.

  2. Soul Man — Sam & Dave (1967)
     Reason: Released during James's reminiscence bump, this song is a classic soul hit that aligns with his cultural background and the vibrant music scene of the 1960s.

  3. Shop Around — The Miracles (featuring Bill "Smokey" Robinson) (1961)
     Reason: This song is by

In [25]:
def run_variant_a(profile):
    """Baseline: no retrieval, just the LLM."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a music therapist. Respond with valid JSON only."},
            {"role": "user", "content": BASELINE_PROMPT.format(
                profile=json.dumps(profile, indent=2)
            )},
        ],
        temperature=0.4,
    )
    text = response.choices[0].message.content
    text = text.strip().strip("```json").strip("```").strip()
    return json.loads(text)

def run_variant_b(profile, faiss_index, bm25, df, method="dense", k=20):
    """RAG with birth-year-only query. Retrieval method and k are tunable."""
    bump_start = profile["birth_year"] + 15
    bump_end = profile["birth_year"] + 25
    query = f"popular songs {bump_start}-{bump_end}"

    retrieved = retrieve(
        query,
        faiss_index,
        df,
        k=k,
        method=method,
        bm25_index=bm25
    )
    result = generate_playlist(profile, retrieved, GENERATION_PROMPT)
    return result, retrieved

def run_variant_c(profile, faiss_index, bm25, df,
                  method="dense", k_per_query=10, total_k=20):
    """RAG with full biographical profiling. Method and k are tunable."""
    retrieved, queries = profile_to_context(
        profile, faiss_index, bm25, df,
        method=method, k_per_query=k_per_query, total_k=total_k
    )
    result = generate_playlist(profile, retrieved, GENERATION_PROMPT)
    return result, retrieved, queries

In [26]:
result_a = run_variant_a(patient)
result_b, retrieved_b = run_variant_b(patient, faiss_index, bm25, df)
result_c, retrieved_c, queries_c = run_variant_c(patient, faiss_index, bm25, df)

print("Variant A playlist top 3:")
for song in result_a["playlist"][:3]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")

print("\nVariant B playlist top 3:")
for song in result_b["playlist"][:3]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")

print("\nVariant C playlist top 3:")
for song in result_c["playlist"][:3]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")

Generated 8 queries:
  - popular Motown hits in Detroit 1963-1973
  - top R&B songs 1966 graduation year
  - Vietnam War era songs popular among African American soldiers
  - wedding songs popular in Detroit 1971
  - soul and funk hits 1968-1973 in Detroit
  - African American male favorite songs 1965-1970
  - hits played on Detroit radio stations 1963-1973
  - songs celebrating fatherhood in 1975
Variant A playlist top 3:
  1. My Girl — The Temptations (1965)
  2. Ain't No Mountain High Enough — Marvin Gaye & Tammi Terrell (1967)
  3. I Heard It Through the Grapevine — Marvin Gaye (1968)

Variant B playlist top 3:
  1. Hot Fun In The Summertime — Sly & The Family Stone (1969)
  2. The Long And Winding Road/For You Blue — The Beatles (1970)
  3. American Pie (Parts I & II) — Don McLean (1972)

Variant C playlist top 3:
  1. Papa Was A Rollin' Stone — The Temptations (1973)
  2. Soul Man — Sam & Dave (1967)
  3. The Ballad Of The Green Berets — SSgt Barry Sadler (1966)


In [27]:
bump_start = patient["birth_year"] + 15
bump_end = patient["birth_year"] + 25

hist_a = historical_plausibility(result_a["playlist"], df, bump_start, bump_end)
hist_b = historical_plausibility(result_b["playlist"], df, bump_start, bump_end)
hist_c = historical_plausibility(result_c["playlist"], df, bump_start, bump_end)

judge_a = llm_judge(patient, result_a["playlist"])
judge_b = llm_judge(patient, result_b["playlist"])
judge_c = llm_judge(patient, result_c["playlist"])

print("=== Historical Plausibility ===")
print("Variant A:", hist_a)
print("Variant B:", hist_b)
print("Variant C:", hist_c)

print("\n=== LLM Judge ===")
print("Variant A:", judge_a)
print("Variant B:", judge_b)
print("Variant C:", judge_c)

=== Historical Plausibility ===
Variant A: 0.7
Variant B: 1.0
Variant C: 0.9

=== LLM Judge ===
Variant A: {'biographical_precision': 5, 'cultural_appropriateness': 5, 'overall_quality': 5, 'reasoning': "The playlist is highly tailored to James Wilson's life events and personal history, featuring songs from his teenage years, significant life events like his marriage, and his early family life. The selection is deeply rooted in the Motown sound, which is culturally appropriate given his African American background and Detroit origins. Overall, the playlist is well-curated to evoke nostalgia and positive emotions, making it an excellent therapeutic tool for James."}
Variant B: {'biographical_precision': 4, 'cultural_appropriateness': 4, 'overall_quality': 4, 'reasoning': "The playlist effectively captures key life events and time periods relevant to James, such as his high school years, marriage, and early parenthood. Songs like 'Hot Fun In The Summertime' and 'The Long And Winding Road